$$
\min \sum_{w \in W} \sum_{t \in T} C_{w,t} x_{w,t}
$$

$$
\sum_{w \in W} x_{w,t} = 1 \quad \forall t \in T
$$

$$
\sum_{t \in T} x_{w,t} \le 1 \quad \forall w \in W
$$
**Sets:** $W$ (workers), $T$ (tasks)  
**Parameters:** $C_{w,t}$ (cost)  
**Variables:** $x_{w,t} \in \{0,1\}$ (assignment)


In [1]:
from pyomo.environ import *
from ortools.sat.python import cp_model

In [34]:
# rows: workers
# cols: tasks
cost_matrix = [
    [90, 76, 75, 70],
    [35, 85, 55, 65],
    [125, 95, 90, 105],
    [45, 110, 95, 115],
]

num_workers = len(cost_matrix)
num_tasks = len(cost_matrix[0])

In [44]:
def lp():
    model = ConcreteModel()
    model.T = Set(initialize=list(range(num_tasks)))
    model.W = Set(initialize=list(range(num_workers)))
    
    # Can use cost_matrix directly too!
    def give_cost(model, w, t):
        return cost_matrix[w][t]
    model.Cost = Param(model.W, model.T, initialize=give_cost)
    
    def obj_rule(model):
        return sum(model.Cost[w,t] * model.x[w,t] for w in model.W for t in model.T)
    
    def one_worker_per_task_rule(model, t):
        return sum(model.x[w,t] for w in model.W) == 1
    
    def one_task_per_worker_rule(model, w):
        return sum(model.x[w,t] for t in model.T) <= 1

    
    model.x = Var(model.W, model.T, domain=Binary)
    model.obj = Objective(rule=obj_rule, sense=minimize)
    model.task_cons = Constraint(model.T, rule=one_worker_per_task_rule)
    model.worker_cons = Constraint(model.T, rule=one_task_per_worker_rule)
    
    solver = SolverFactory("glpk")
    results = solver.solve(model)
    print(value(model.obj()))

In [45]:
lp()

265.0


In [46]:
x = {}
model = cp_model.CpModel()

for w in range(num_workers):
    for t in range(num_tasks):
        x[w,t] = model.NewBoolVar(f"x{w}{t}")

        
#one worker per task
for t in range(num_tasks):
    # This is equivalent to model.Add(sum(list) == 1),
    # but it only works for boolean (binary) variables.
    model.AddExactlyOne([x[w,t] for w in range(num_workers)])

# one task per worker 
for w in range(num_workers):
    # This is equivalent to model.Add(sum(list) <= 1),
    # but it only works for boolean (binary) variables.
    model.AddAtMostOne([x[w,t] for t in range(num_tasks)])

In [47]:
costs = []
variables = []

for w in range(num_workers):
    for t in range(num_tasks):
        variables.append(x[w,t])
        costs.append(cost_matrix[w][t])

model.Minimize(cp_model.LinearExpr.WeightedSum(variables, costs))
# model.Minimize(cp_model.LinearExpr.Sum([cost_matrix[w][t]*x[w,t] for w in range(num_workers) for t in range(num_tasks)]))

In [48]:
solver = cp_model.CpSolver()
results = solver.Solve(model)

In [50]:
print(results)
print(solver.ObjectiveValue())
# for w in range(num_workers):
    # for t in range(num_tasks):
        # print(x[w,t],solver.Value(x[w,t]))

CpSolverStatus.OPTIMAL
265.0
